In [1]:
import os
import time
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from huggingface_hub import get_collection
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from google import genai
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

C:\Users\Personal\PycharmProjects\NLP-RAG-BY-HOPETOSKILL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Personal\AppData\Local\Temp\ipykernel_5564\3777141342.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2113.24it/s]


In [3]:
parser = StrOutputParser()

In [4]:
#all api keys

#QDRANT
QDRANT_API_KEY = os.getenv("QDRANTAPIKEY")
QDRANT_ENDPOINT = os.getenv("QDRANTENDPOINT")

#gemini model apikey
geminiapikey = os.getenv("GEMINIAPIKEY")

#
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash-lite",
#     google_api_key=geminiapikey,
#     temperature=0
# )


groq_api_key = os.getenv("GROQAPIKEY")  # make sure this matches your .env variable name

llm = ChatGroq(
    model="qwen/qwen3-32b",
    api_key=groq_api_key,
    temperature=0
)

In [5]:
from langchain_community.document_loaders import PyPDFLoader

In [6]:
file_path = "who.pdf"
loader = PyPDFLoader(file_path)

document = loader.load()

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)


In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=150,
    separators=[
        "\n\n",   # paragraphs (highest priority)
        "\n",     # lines
        ". ",     # sentences
        " ",      # words
        ""        # fallback
    ]
)

chunks = splitter.split_documents(document)

In [8]:
print(f"all pages in document are {len(document)}")
print(f"chunks created are {len(chunks)}")

all pages in document are 14
chunks created are 203


In [9]:
client = QdrantClient(
       url=QDRANT_ENDPOINT,
       api_key=QDRANT_API_KEY

)

In [12]:
collections = client.get_collections().collections


#Any go through list and stop at true(meet your condition)
collection_exist = any(
    collection.name == "who_dense_only"
    for collection in collections
)

if not collection_exist:
    vectorestore = QdrantVectorStore.from_documents(
        documents=chunks,
        api_key=QDRANT_API_KEY,
        url=QDRANT_ENDPOINT,
        embedding=embeddings,
        collection_name="who_dense_only"
    )
    print("New Collection Created")

else:
    vectorestore = QdrantVectorStore.from_existing_collection(
        collection_name="who_dense_only",
        api_key=QDRANT_API_KEY,
        url=QDRANT_ENDPOINT,
        embedding=embeddings
    )

    print("Collection already exists. Using existing collection.")

Collection already exists. Using existing collection.


In [13]:
#dense search

query = "3-fluorophenmetrazine"
results = vectorestore.similarity_search(query, k=3)

for i,doc in enumerate(results):
    print(f"{i+1},{doc.page_content}")

1,. 6 Benzylone .................................................................................................................................. 7 3-fluorophenmetrazine (3-FPM) ................................................................................................
2,. 7 3-fluorophenmetrazine (3-FPM) ................................................................................................. 7 Opioids .........................................................................................................................................
3,. 6 4-Fluoromethcathinone (flephedrone; 4-FMC) ........................................................................... 6 Benzylone .................................................................................................................................


In [15]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

In [18]:

#Hybrid search

bm25_retriever = BM25Retriever.from_documents(
    documents=chunks
)
# Set the number of documents to retrieve
bm25_retriever.k = 3

qdrant_retriever = vectorestore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

hybrid_retriever = EnsembleRetriever(
    retrievers=[qdrant_retriever, bm25_retriever],
    weights=[0.5, 0.5],  # Equal weight: 50% dense, 50% sparse
)




In [20]:
query1 = "3-fluorophenmetrazine"


dense_results = qdrant_retriever.invoke(query1)
sparse_results = bm25_retriever.invoke(query1)
hybrid_results = hybrid_retriever.invoke(query1)

print("Dense (Qdrant) - Top result:")
has_match = query1.lower() in dense_results[0].page_content.lower()
print(f"  {'[EXACT MATCH]' if has_match else '[No exact match]'}")
print(f"  {dense_results[0].page_content[:120]}...\n")

print("Sparse (BM25) - Top result:")
has_match = query1.lower() in sparse_results[0].page_content.lower()
print(f"  {'[EXACT MATCH]' if has_match else '[No exact match]'}")
print(f"  {sparse_results[0].page_content[:120]}...\n")

print("Hybrid (Ensemble) - Top result:")
has_match = query1.lower() in hybrid_results[0].page_content.lower()
print(f"  {'[EXACT MATCH]' if has_match else '[No exact match]'}")
print(f"  {hybrid_results[0].page_content[:120]}...\n")

Dense (Qdrant) - Top result:
  [EXACT MATCH]
  . 6 Benzylone .............................................................................................................

Sparse (BM25) - Top result:
  [EXACT MATCH]
  . 6 Benzylone .............................................................................................................

Hybrid (Ensemble) - Top result:
  [EXACT MATCH]
  . 6 Benzylone .............................................................................................................



In [21]:
query2 = "Which substances cause opioid-like effects?"

dense_results2 = qdrant_retriever.invoke(query2)
sparse_results2 = bm25_retriever.invoke(query2)
hybrid_results2 = hybrid_retriever.invoke(query2)

print("Dense (Qdrant) - Top result:")
print(f"  {dense_results2[0].page_content[:120]}...\n")

print("Sparse (BM25) - Top result:")
print(f"  {sparse_results2[0].page_content[:120]}...\n")

print("Hybrid (Ensemble) - Top result:")
print(f"  {hybrid_results2[0].page_content[:120]}...\n")

print("=" * 70)
print("CONCLUSION: Why Use EnsembleRetriever?")
print("=" * 70)
print("The Hybrid retriever (EnsembleRetriever) combines strengths:")
print("  1. Finds exact matches (like BM25)")
print("  2. Understands meaning (like dense search)")
print("  3. Works with ANY LangChain retriever")
print("  4. Production-ready with automatic RRF fusion")

Dense (Qdrant) - Top result:
  . Opioid effects can be produced by consumption of the plant, extracts of the plant or by the purified substances. Use o...

Sparse (BM25) - Top result:
  . It produces opioid-like effects with adverse effects including respiratory depression, nausea, dizziness, and vomiting...

Hybrid (Ensemble) - Top result:
  . Opioid effects can be produced by consumption of the plant, extracts of the plant or by the purified substances. Use o...

CONCLUSION: Why Use EnsembleRetriever?
The Hybrid retriever (EnsembleRetriever) combines strengths:
  1. Finds exact matches (like BM25)
  2. Understands meaning (like dense search)
  3. Works with ANY LangChain retriever
  4. Production-ready with automatic RRF fusion
